# Unit 2.2: Edge Detection

This notebook demonstrates various edge detection techniques:
- Sobel operator
- Canny edge detection
- Laplacian edge detection
- Prewitt operator
- Performance comparison

## Import Libraries

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.ndimage import convolve
import warnings
warnings.filterwarnings('ignore')

## 1. Load Image

In [ ]:
# Load image
img_bgr = cv2.imread('sample.jpeg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# Apply slight blur to reduce noise
img_blurred = cv2.GaussianBlur(img_gray, (5, 5), 1.0)

print(f"Image Shape: {img_gray.shape}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original Grayscale')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_blurred, cmap='gray')
plt.title('Blurred (Preprocessing)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Sobel Edge Detection

In [ ]:
# Sobel Kernels
sobel_x = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]], dtype=np.float32)

sobel_y = np.array([[-1, -2, -1],
                   [ 0,  0,  0],
                   [ 1,  2,  1]], dtype=np.float32)

# Apply Sobel operator using OpenCV
sobel_x_cv = cv2.Sobel(img_blurred, cv2.CV_32F, 1, 0, ksize=3)
sobel_y_cv = cv2.Sobel(img_blurred, cv2.CV_32F, 0, 1, ksize=3)

# Calculate gradient magnitude and direction
gradient_magnitude = np.sqrt(sobel_x_cv**2 + sobel_y_cv**2)
gradient_direction = np.arctan2(sobel_y_cv, sobel_x_cv) * 180 / np.pi

# Normalize for display
gradient_magnitude_normalized = np.clip(gradient_magnitude, 0, 255).astype(np.uint8)

# Apply threshold
sobel_binary = cv2.threshold(gradient_magnitude_normalized, 100, 255, cv2.THRESH_BINARY)[1]

print(f"Gradient Magnitude Range: {gradient_magnitude.min():.2f} to {gradient_magnitude.max():.2f}")
print(f"Gradient Direction Range: {gradient_direction.min():.2f}° to {gradient_direction.max():.2f}°")

# Visualize
plt.figure(figsize=(15, 4))

plt.subplot(1, 4, 1)
plt.imshow(img_blurred, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(gradient_magnitude_normalized, cmap='gray')
plt.title('Gradient Magnitude')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(gradient_direction, cmap='hsv')
plt.title('Gradient Direction')
plt.colorbar(label='Angle (degrees)')

plt.subplot(1, 4, 4)
plt.imshow(sobel_binary, cmap='gray')
plt.title('Binary Edges (Threshold=100)')
plt.axis('off')

plt.tight_layout()
plt.show()

### 2.1 Sobel Components Visualization

In [ ]:
# Visualize X and Y components separately
sobel_x_normalized = np.clip(np.abs(sobel_x_cv), 0, 255).astype(np.uint8)
sobel_y_normalized = np.clip(np.abs(sobel_y_cv), 0, 255).astype(np.uint8)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(sobel_x_normalized, cmap='gray')
plt.title('Sobel-X (Vertical Edges)')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(sobel_y_normalized, cmap='gray')
plt.title('Sobel-Y (Horizontal Edges)')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(gradient_magnitude_normalized, cmap='gray')
plt.title('Combined Magnitude\n√(Gx² + Gy²)')
plt.axis('off')

plt.tight_layout()
plt.show()

## 3. Canny Edge Detection

In [ ]:
# Canny edge detection with different thresholds
threshold_pairs = [(50, 150), (100, 200), (150, 255)]
canny_results = []

plt.figure(figsize=(15, 4))

plt.subplot(1, 4, 1)
plt.imshow(img_blurred, cmap='gray')
plt.title('Original')
plt.axis('off')

for idx, (thresh_low, thresh_high) in enumerate(threshold_pairs):
    # Apply Canny edge detection
    edges = cv2.Canny(img_blurred, thresh_low, thresh_high)
    canny_results.append(edges)
    
    plt.subplot(1, 4, idx + 2)
    plt.imshow(edges, cmap='gray')
    plt.title(f'Canny ({thresh_low}, {thresh_high})')
    plt.axis('off')

plt.tight_layout()
plt.show()

print("Canny Edge Detection Steps:")
print("1. Gaussian blur to reduce noise")
print("2. Calculate gradient magnitude and direction")
print("3. Non-maximum suppression")
print("4. Double thresholding")
print("5. Edge tracking by hysteresis")

## 4. Laplacian Edge Detection

In [ ]:
# Laplacian kernels (different variations)
laplacian_kernel_1 = np.array([[0, 1, 0],
                               [1, -4, 1],
                               [0, 1, 0]], dtype=np.float32)

laplacian_kernel_2 = np.array([[1, 1, 1],
                               [1, -8, 1],
                               [1, 1, 1]], dtype=np.float32)

# Apply Laplacian using OpenCV
laplacian_cv = cv2.Laplacian(img_blurred, cv2.CV_32F, ksize=3)
laplacian_normalized = np.clip(np.abs(laplacian_cv), 0, 255).astype(np.uint8)

# Apply threshold
laplacian_binary = cv2.threshold(laplacian_normalized, 50, 255, cv2.THRESH_BINARY)[1]

plt.figure(figsize=(15, 4))

plt.subplot(1, 4, 1)
plt.imshow(img_blurred, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(laplacian_kernel_1, cmap='hot')
plt.title('Laplacian Kernel 1')
plt.colorbar()

plt.subplot(1, 4, 3)
plt.imshow(laplacian_normalized, cmap='gray')
plt.title('Laplacian Output (Absolute)')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(laplacian_binary, cmap='gray')
plt.title('Binary Edges (Threshold=50)')
plt.axis('off')

plt.tight_layout()
plt.show()

## 5. Prewitt Edge Detection

In [ ]:
# Prewitt kernels
prewitt_x = np.array([[-1, 0, 1],
                      [-1, 0, 1],
                      [-1, 0, 1]], dtype=np.float32)

prewitt_y = np.array([[-1, -1, -1],
                      [ 0,  0,  0],
                      [ 1,  1,  1]], dtype=np.float32)

# Apply Prewitt using scipy
prewitt_x_result = convolve(img_blurred.astype(np.float32), prewitt_x)
prewitt_y_result = convolve(img_blurred.astype(np.float32), prewitt_y)

prewitt_magnitude = np.sqrt(prewitt_x_result**2 + prewitt_y_result**2)
prewitt_normalized = np.clip(prewitt_magnitude, 0, 255).astype(np.uint8)
prewitt_binary = cv2.threshold(prewitt_normalized, 50, 255, cv2.THRESH_BINARY)[1]

plt.figure(figsize=(15, 4))

plt.subplot(1, 4, 1)
plt.imshow(img_blurred, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(prewitt_x, cmap='hot')
plt.title('Prewitt-X Kernel')
plt.colorbar()

plt.subplot(1, 4, 3)
plt.imshow(prewitt_normalized, cmap='gray')
plt.title('Prewitt Magnitude')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(prewitt_binary, cmap='gray')
plt.title('Binary Edges (Threshold=50)')
plt.axis('off')

plt.tight_layout()
plt.show()

## 6. Comprehensive Edge Detection Comparison

In [ ]:
# Prepare all edge detection results
all_edges = {
        'Original': img_blurred,
        'Sobel': sobel_binary,
        'Canny\n(100,200)': canny_results[1],
        'Laplacian': laplacian_binary,
        'Prewitt': prewitt_binary
}

# Create comparison
plt.figure(figsize=(15, 3))

for idx, (name, edge_img) in enumerate(all_edges.items()):
    plt.subplot(1, 5, idx + 1)
    plt.imshow(edge_img, cmap='gray')
    plt.title(name, fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.suptitle('Edge Detection Methods Comparison', fontsize=14, y=1.02)
plt.show()

print("\nEdge Detection Statistics\n")
print(f"{'Method':<20} {'Edges Found':<15} {'Edge Percentage':<15}")
print("-" * 50)

for name, edge_img in list(all_edges.items())[1:]:  # Skip original
    if edge_img is not None:
        edge_count = np.sum(edge_img > 0)
        total_pixels = edge_img.shape[0] * edge_img.shape[1]
        percentage = (edge_count / total_pixels) * 100
        print(f"{name:<20} {edge_count:<15} {percentage:<15.2f}%")

## 7. Custom Edge Detection Implementation

In [ ]:
def manual_edge_detection(image, method='sobel'):
        """
        Manual edge detection implementation
        
        Args:
            image: Grayscale image
            method: 'sobel', 'prewitt', or 'laplacian'
        
        Returns:
            Edge magnitude image
        """
        if method == 'sobel':
            sobelX = cv2.Sobel(image, cv2.CV_32F, 1, 0, ksize=3)
            sobelY = cv2.Sobel(image, cv2.CV_32F, 0, 1, ksize=3)
            magnitude = np.sqrt(sobelX**2 + sobelY**2)
        
        elif method == 'prewitt':
            prewitt_x = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
            prewitt_y = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float32)
            
            prewittX = convolve(image.astype(np.float32), prewitt_x)
            prewittY = convolve(image.astype(np.float32), prewitt_y)
            magnitude = np.sqrt(prewittX**2 + prewittY**2)
        
        elif method == 'laplacian':
            magnitude = cv2.Laplacian(image, cv2.CV_32F, ksize=3)
            magnitude = np.abs(magnitude)
        
        return magnitude

# Test custom implementation
test_methods = ['sobel', 'prewitt', 'laplacian']
custom_results = {}

plt.figure(figsize=(15, 3))

for idx, method in enumerate(test_methods):
    magnitude = manual_edge_detection(img_blurred, method=method)
    magnitude_normalized = np.clip(magnitude, 0, 255).astype(np.uint8)
    custom_results[method] = magnitude_normalized
    
    plt.subplot(1, 3, idx + 1)
    plt.imshow(magnitude_normalized, cmap='gray')
    plt.title(f'Custom {method.capitalize()}')
    plt.axis('off')

plt.tight_layout()
plt.show()

print("Custom edge detection implementations verified")

## 8. Real-world Applications

In [ ]:
# Create synthetic test cases
# Test case 1: Simple geometric shapes
test_img1 = np.zeros((200, 200), dtype=np.uint8)
cv2.rectangle(test_img1, (50, 50), (150, 150), 255, -1)
cv2.circle(test_img1, (100, 100), 30, 0, -1)

# Add noise
test_img1_noisy = cv2.GaussianBlur(test_img1, (3, 3), 0.5)

edges_canny_test = cv2.Canny(test_img1_noisy, 30, 100)
sobel_test = cv2.Sobel(test_img1_noisy, cv2.CV_32F, 1, 0, ksize=3)
sobel_test = np.clip(np.abs(sobel_test), 0, 255).astype(np.uint8)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(test_img1, cmap='gray')
plt.title('Test Image\n(Geometric Shapes)')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(edges_canny_test, cmap='gray')
plt.title('Canny Edges')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(sobel_test, cmap='gray')
plt.title('Sobel Edges')
plt.axis('off')

plt.tight_layout()
plt.suptitle('Edge Detection Applications', fontsize=14, y=1.02)
plt.show()

## Summary & Key Points

### Edge Detection Methods:

1. **Sobel**
   - First-order derivative
   - Good for general edge detection
   - Moderate noise sensitivity

2. **Canny**
   - Best for accurate edge localization
   - Low noise sensitivity
   - Multi-stage process

3. **Laplacian**
   - Second-order derivative
   - Sensitive to noise
   - Detects all edges

4. **Prewitt**
   - Similar to Sobel
   - Different kernel weights
   - Good edge detection

### Best Practices:
- Always apply Gaussian blur first to reduce noise
- Use Canny for critical applications
- Adjust thresholds based on image content
- Consider edge direction/orientation when needed